In [7]:
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

from sklearn.compose import ColumnTransformer

import pandas as pd
import numpy as np



# 1. Load data
housing = pd.read_csv('california_housing_data.csv')



#------------------------------


# 2. Create a stratified test set based on income category


# Create new column "income categories" -
# Acting as STRATA where specific attribut is "median_income" colm
housing[ 'income_categories'] = pd.cut( housing['median_income'] ,
                                       bins=[ 0 ,2 ,4, 6 , 8, np.inf],
                                        labels=[ 1 ,2 , 3, 4, 5 ] )


# Assume income_cat.. is a column in the dataset created from median_income
shuffle_split = StratifiedShuffleSplit( n_splits=1 , test_size=0.2 , random_state=42)

# This ensures that the income distribution in both sets is similar to that of the full dataset
for test_split_indx , train_split_indx in shuffle_split.split( housing , housing[ 'income_categories'] ):
    test_data = housing.loc[test_split_indx]
    train_data = housing.loc[train_split_indx]
    
    
#removing strata cols
train_data.drop( columns=['income_categories'] , inplace = True)   
test_data.drop( columns=['income_categories'] , inplace = True)  


# Copying the training data to work on
housing = train_data.copy()



#---------------------------------


# 3. Seperating feature and label from train data

housing_label = housing['median_house_value'].copy()

housing_features = housing.drop( columns=['median_house_value'] , axis=1)


#-------------------------------

# 4. Separate numerical and categorical columns names in [ list ]- helps in making 2 pipeines

nums_attributes = housing_features.drop(columns=['ocean_proximity'] , axis=1).columns.to_list()

catg_attribute = ['ocean_proximity']


#-------------------------------

# 5. Pipelines


# Numerical pipeline for nums_attribute
nums_pipeline = Pipeline([
    ('imputer' , SimpleImputer(strategy='median')),
    ('scaler' , StandardScaler())
])


# Catogarical Pipeline for catg_attribute
catg_pipeline = Pipeline([
    # ('encoder' , OriginalEncoder()),  -> if import it
    ('onehot_enc' , OneHotEncoder( handle_unknown='ignore'))   # handle_unknown -> if any new/catd data comes in test data then it (ignore it by making it 0 numerialy)
])

# ------------------------


# 6. Construct Full Pipeline - by ColumnTransformer


# ColumnTransformer -> Applies different preprocessing pipelines to different columns( nums/catogary)

# It applies transformations in parallel to different feature columns

# params: list of tuples (name, transformer, columns)
# - name: identifier for the transformer
# - transformer: pipeline or transformer object to apply
# - columns: list of column names to apply the transformer to

# remainder='passthrough' keeps other columns as-is

# sparse_threshold=0 ensures dense output

full_pipeline = ColumnTransformer([
    ('nums' , nums_pipeline , nums_attributes),    # (name , pipeline , attribte_of that pipeline type)
    ('catg' , catg_pipeline , catg_attribute)
])


# Apply the pipeline of data -> to get final train data but in numpy data
prepared_housing_numpy = full_pipeline.fit_transform(housing_features)




#-----------------------------



# 7. Converting prepared_housing_numpy into dataframe data

housing_prepared = pd.DataFrame(prepared_housing_numpy ,
                                columns=['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 
                                         'households', 'median_income','INLAND','NEAR BAY','NEAR OCEAN','<1H OCEAN'] ,
                                index=housing_features.index)


print( housing_prepared)

       longitude  latitude  housing_median_age  total_rooms  total_bedrooms  \
3700    0.600263 -0.668257           -1.424234     0.141814        0.367324   
20333   0.285900 -0.649575           -1.583145     3.559411        2.554428   
19723  -1.051392  1.508263            1.356712    -0.868768       -0.932735   
14228   1.258930 -1.373525           -1.185867    -0.569057       -0.606547   
11027   0.864728 -0.850413            0.164878    -0.260021       -0.280359   
...          ...       ...                 ...          ...             ...   
10739   0.819819 -0.957837            0.641612     0.334517       -0.085584   
11300   0.819819 -0.864424           -0.073489     0.449517        1.012661   
6766    0.715031 -0.710293           -0.391311     0.918399        1.540663   
10281   0.869718 -0.808377           -0.629678     0.211969       -0.045691   
4768    0.600263 -0.738317            0.403245     0.283011        0.651272   

       population  households  median_income  INLAN

-------------------
<br>
<br>

# Training ML Algorithms on Preprocessed data

Now that our data is preprocessed, let’s move on to training machine learning models and evaluating their performance. We'll start with:

 1. Linear Regression   <br>
 2. Decision Tree Regressor  <br>
 3. Random Forest Regressor   <br>

- We’ll first test them on the training data and then use cross-validation to get a better estimate of their true performance.

=========================

## 1. Train and Test Models on the Training Set

In [8]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error



# Linear Regression
lin_reg = LinearRegression()
lin_reg.fit(housing_prepared, housing_label)

# Decision Tree
tree_reg = DecisionTreeRegressor()
tree_reg.fit(housing_prepared, housing_label)

# Random Forest
forest_reg = RandomForestRegressor()
forest_reg.fit(housing_prepared, housing_label)



# Predict using training data
lin_preds = lin_reg.predict(housing_prepared)
tree_preds = tree_reg.predict(housing_prepared)
forest_preds = forest_reg.predict(housing_prepared)


# Calculate RMSE
lin_rmse = root_mean_squared_error(housing_label, lin_preds)
tree_rmse = root_mean_squared_error(housing_label, tree_preds)
forest_rmse = root_mean_squared_error(housing_label, forest_preds)

print("Linear Regression RMSE:", lin_rmse)
print("Decision Tree RMSE:", tree_rmse)
print("Random Forest RMSE:", forest_rmse)

Linear Regression RMSE: 69368.48218767327
Decision Tree RMSE: 0.0
Random Forest RMSE: 20928.245859037837



--------------------------------------------------------------------------------------
<br>

## A Warning About Training RMSE
    Training RMSE only shows how well the model fits the training data. 
    It does not tell us how well it will perform on unseen data. 
    In fact, the Decision Tree and Random Forest may overfit, leading to very low training error but poor generalization.


--------------------------------------------------------------------------------------
<br>

# 2. Cross-Validation: A Better Evaluation Strategy
Cross-validation helps us evaluate how a model generalizes to new data without needing to touch the test set.

## What is Cross-Validation?
Instead of training the model once and evaluating on a holdout set, k-fold cross-validation splits the training data into k folds (typically 10), trains the model on k-1 folds, and validates it on the remaining fold. This process repeats k times.

- We’ll use cross_val_score from sklearn.model_selection.

### Cross-Validation on Decision Tree

In [9]:
from sklearn.model_selection import cross_val_score
import pandas as pd

# Evaluate Decision Tree with cross-validation
tree_rmses = -cross_val_score(
    tree_reg,
    housing_prepared,
    housing_label,
    scoring="neg_root_mean_squared_error",
    cv=10
)

# WARNING: Scikit-Learn’s scoring uses utility functions (higher is better), so RMSE is returned as negative.
# We use minus (-) to convert it back to positive RMSE.

print("Decision Tree CV RMSEs:", tree_rmses)

print("\nCross-Validation Performance (Decision Tree):")

print(pd.Series(tree_rmses).describe())

Decision Tree CV RMSEs: [82975.82527872 74848.56755428 74490.88041457 74814.0616359
 75508.63407522 70779.73598909 79421.95169912 69744.47953227
 72571.69212306 81817.84589763]

Cross-Validation Performance (Decision Tree):
count       10.000000
mean     75697.367420
std       4430.832306
min      69744.479532
25%      73051.489196
50%      74831.314595
75%      78443.622293
max      82975.825279
dtype: float64


# WARNING: Scikit-Learn’s scoring uses utility functions (higher is better), so RMSE is returned as negative.
## We use minus (-) to convert it back to positive RMSE.

-------------------------------



# WE CHOOSE THE RANDOM FOREST AS MODEL